**<h1 style="text-align: center;font-size: 3rem">Feature Engineering</h1><h2 style="text-align: center;font-size: 1.3rem">(Notebook II)</h2>**


In [ ]:
from os import getenv

import numpy as np
import pandas as pd
from dotenv import load_dotenv

from fraud_detection_pipeline.data_loading import tts_csv
from fraud_detection_pipeline.utils.constants import (
    PARENT_DIR,
    RAW_DIRECTORY,
    PROCESSED_DIRECTORY,
)

In [ ]:
load_dotenv()
RANDOM_STATE = int(getenv("RANDOM_STATE", 0))
TEST_SPLIT_SIZE = float(getenv("TEST_SPLIT_SIZE", 0.3))

In [ ]:
transactions = pd.read_csv(PARENT_DIR / RAW_DIRECTORY / "creditcard.csv")

## Minor Column Changes

Column names are converted from their capitalized format to an all lowercase format.


For improved interpretation of the dataset, the _'Class'_ column (renamed to _'class'_ in the step above) is renamed again to _'is_fraud'_.

This helps associate 1 values (positive) to indicate a fraudulent transaction and associate 0 values (negative) to indicate a genuine transaction.


In [ ]:
def renamer(name: str) -> str:
    name = str(name).lower().replace(" ", "_")
    match name:
        case "class":
            return "is_fraud"
        case "time":
            return "time_elapsed"
        case _:
            return name


transactions = transactions.rename(columns=renamer)

print("\n".join(transactions.columns))

## Transforming Existing Features


### Cyclic Time Features


Cyclic time features provide potential insight in routinely rhythm. This can be helpful if fraudulent transactions happen during a certain time of day.


In [ ]:
seconds_in_a_day = 60 * 60 * 24

In [ ]:
time_elapsed = transactions["time_elapsed"]

In [ ]:
print(f"Days: {time_elapsed.max() / (seconds_in_a_day)}")

The transactions are recorded over a time period of 2 days. Cyclic features would be influenced by the hour of the day, seeing if the time of day may influence whether a transaction is likely to be fraudulent.


In [ ]:
transactions["hour_sin"] = np.sin(2 * np.pi * time_elapsed / seconds_in_a_day)
transactions["hour_cos"] = np.cos(2 * np.pi * time_elapsed / seconds_in_a_day)

In [ ]:
transactions[["hour_sin", "hour_cos"]].head(10)

## Type Transformations

As the _'time'_ feature (previously _'Time'_) representing the time elapsed since the first transaction is measured in 1 second intervals, the need to store it as a float which is more appropriate for continuous data is irrelevant. The column's values are casted as a 32-bit integer.

The _'is_fraud'_ feature (previously _'Class'_) is either _1_ to indicate a fraudulent transaction or _0_ to indicate a genuine transaction. Storing this binary value as a 64-bit integer is unnecessary so the type is casted to a boolean.


In [ ]:
transactions["time_elapsed"] = transactions["time_elapsed"].astype("int64")
transactions["is_fraud"] = transactions["is_fraud"].astype("int32")
print(transactions.dtypes)

In [ ]:
transactions = transactions.drop(columns=["time_elapsed"])

In [ ]:
X = transactions.drop(columns=["is_fraud"])
y = transactions["is_fraud"]

In [ ]:
X.shape, y.shape

Unfortunately, since variables v1 to v28 are unknown in their subject and nature, only that they are continuous values that appear to potentially have normalization performed before ingestion. Additional feature transformations cannot be made.


In [ ]:
transactions["is_fraud"] = transactions.pop("is_fraud")

In [ ]:
transactions.to_csv(
    PARENT_DIR / PROCESSED_DIRECTORY / "creditcard.csv",
    index=False,
)

In [ ]:
X_train, X_test, y_train, y_test = tts_csv(
    PARENT_DIR / PROCESSED_DIRECTORY / "creditcard.csv",
    target="is_fraud",
    test_size=TEST_SPLIT_SIZE,
    random_state=RANDOM_STATE,
)

In [ ]:
pd.concat([X_train, y_train], axis=1).to_csv(
    PARENT_DIR / PROCESSED_DIRECTORY / "train.csv",
    index=False,
)
pd.concat([X_test, y_test], axis=1).to_csv(
    PARENT_DIR / PROCESSED_DIRECTORY / "test.csv",
    index=False,
)